In [2]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
BASE_PATH = "/content/drive/MyDrive/NetShield-AI"

print(BASE_PATH)

/content/drive/MyDrive/NetShield-AI


In [4]:
!pip install -q pandas numpy scikit-learn scipy joblib matplotlib seaborn plotly

In [5]:
import pandas as pd
train_path = f"{BASE_PATH}/data/raw/UNSW_NB15_training-set.csv"
test_path = f"{BASE_PATH}/data/raw/UNSW_NB15_testing-set.csv"

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

print("Training shape:", train.shape)
print("Testing shape :", test.shape)

Training shape: (175341, 45)
Testing shape : (82332, 45)


In [6]:
train.head()

,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,0.121478,tcp,-,FIN,6,4,258,172,74.087490,...,1,1,0,0,0,1,1,0,Normal,0
1,2,0.649902,tcp,-,FIN,14,38,734,42014,78.473372,...,1,2,0,0,0,1,6,0,Normal,0
2,3,1.623129,tcp,-,FIN,8,16,364,13186,14.170161,...,1,3,0,0,0,2,6,0,Normal,0
3,4,1.681642,tcp,ftp,FIN,12,12,628,770,13.677108,...,1,3,1,1,0,2,1,0,Normal,0
4,5,0.449454,tcp,-,FIN,10,6,534,268,33.373826,...,1,40,0,0,0,2,39,0,Normal,0


In [7]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 175341 entries, 0 to 175340
Data columns (total 45 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   id                 175341 non-null  int64  
 1   dur                175341 non-null  float64
 2   proto              175341 non-null  object 
 3   service            175341 non-null  object 
 4   state              175341 non-null  object 
 5   spkts              175341 non-null  int64  
 6   dpkts              175341 non-null  int64  
 7   sbytes             175341 non-null  int64  
 8   dbytes             175341 non-null  int64  
 9   rate               175341 non-null  float64
 10  sttl               175341 non-null  int64  
 11  dttl               175341 non-null  int64  
 12  sload              175341 non-null  float64
 13  dload              175341 non-null  float64
 14  sloss              175341 non-null  int64  
 15  dloss              175341 non-null  int64  
 16  si

In [8]:
train.columns.tolist()

['id',
 'dur',
 'proto',
 'service',
 'state',
 'spkts',
 'dpkts',
 'sbytes',
 'dbytes',
 'rate',
 'sttl',
 'dttl',
 'sload',
 'dload',
 'sloss',
 'dloss',
 'sinpkt',
 'dinpkt',
 'sjit',
 'djit',
 'swin',
 'stcpb',
 'dtcpb',
 'dwin',
 'tcprtt',
 'synack',
 'ackdat',
 'smean',
 'dmean',
 'trans_depth',
 'response_body_len',
 'ct_srv_src',
 'ct_state_ttl',
 'ct_dst_ltm',
 'ct_src_dport_ltm',
 'ct_dst_sport_ltm',
 'ct_dst_src_ltm',
 'is_ftp_login',
 'ct_ftp_cmd',
 'ct_flw_http_mthd',
 'ct_src_ltm',
 'ct_srv_dst',
 'is_sm_ips_ports',
 'attack_cat',
 'label']

In [9]:
train.isnull().sum()

,0
id,0
dur,0
proto,0
service,0
state,0
spkts,0
dpkts,0
sbytes,0
dbytes,0
rate,0


In [10]:
train["label"].value_counts()

,count
label,
1,119341
0,56000


In [11]:
train["attack_cat"].value_counts(dropna=False)

,count
attack_cat,
Normal,56000
Generic,40000
Exploits,33393
Fuzzers,18184
DoS,12264
Reconnaissance,10491
Analysis,2000
Backdoor,1746
Shellcode,1133


In [12]:
# Remove target columns and the ID column
X_train = train.drop(columns=["id", "label", "attack_cat"])
y_train = train["label"]

X_test = test.drop(columns=["id", "label", "attack_cat"])
y_test = test["label"]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

X_train: (175341, 42)
y_train: (175341,)
X_test : (82332, 42)
y_test : (82332,)


In [13]:
import numpy as np
print("Infinite values in train:",
      np.isinf(X_train.select_dtypes(include=np.number)).sum().sum())

print("Infinite values in test:",
      np.isinf(X_test.select_dtypes(include=np.number)).sum().sum())

Infinite values in train: 0
Infinite values in test: 0


In [14]:
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

In [15]:
categorical_cols = ["proto", "service", "state"]

numerical_cols = [
    col for col in X_train.columns
    if col not in categorical_cols
]

print("Categorical columns:")
print(categorical_cols)

print("\nNumber of numerical columns:", len(numerical_cols))
print("Number of categorical columns:", len(categorical_cols))

Categorical columns:
['proto', 'service', 'state']

Number of numerical columns: 39
Number of categorical columns: 3


In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numerical_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

In [17]:
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

In [18]:
logistic_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['dur', 'spkts', 'dpkts',
                                                   'sbytes', 'dbytes', 'rate',
                                                   'sttl', 'dttl', 'sload',
                                                   'dload', 'sloss', 'dloss',
                                                   'sinpkt', 'dinpkt', 'sjit',
                                                   'djit', 'swin', 'stcpb',
                                                   'dtcpb', 'dwin', 'tcprtt',
                                                   'synack', 'ackdat', 'smean',
                                                   'dmean', 'trans_depth',
                                                   'response_body_len',
                                                   'ct_srv_src', 'ct_state_ttl',
                                                   'ct_dst_ltm', ...]),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['proto', 'service',
                                                   'state'])])),
                ('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

In [19]:
y_pred = logistic_model.predict(X_test)

print(y_pred[:20])

[1 1 1 1 1 1 1 1 0 0 0 0 1 1 1 1 1 1 1 1]


In [20]:
y_prob = logistic_model.predict_proba(X_test)[:, 1]

print(y_prob[:10])

[7.71889770e-01 7.49138833e-01 7.37741995e-01 7.78852523e-01
 7.39917179e-01 8.25197936e-01 8.03930973e-01 6.92554039e-01
 8.73380580e-10 8.73380041e-10]


In [21]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

# Basic metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

# Probability-based metrics
roc_auc = roc_auc_score(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)

print("===== NETSHIELD AI - LOGISTIC REGRESSION =====")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")
print(f"PR-AUC   : {pr_auc:.4f}")

print("\n===== CONFUSION MATRIX =====")
print(confusion_matrix(y_test, y_pred))

print("\n===== CLASSIFICATION REPORT =====")
print(classification_report(y_test, y_pred))

===== NETSHIELD AI - LOGISTIC REGRESSION =====
Accuracy : 0.8355
Precision: 0.8019
Recall   : 0.9313
F1 Score : 0.8618
ROC-AUC  : 0.9561
PR-AUC   : 0.9668

===== CONFUSION MATRIX =====
[[26571 10429]
 [ 3115 42217]]

===== CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

           0       0.90      0.72      0.80     37000
           1       0.80      0.93      0.86     45332

    accuracy                           0.84     82332
   macro avg       0.85      0.82      0.83     82332
weighted avg       0.84      0.84      0.83     82332



In [22]:
import joblib

model_path = f"{BASE_PATH}/models/logistic_binary.pkl"

joblib.dump(logistic_model, model_path)

print("Model saved to:", model_path)

Model saved to: /content/drive/MyDrive/NetShield-AI/models/logistic_binary.pkl


In [23]:
from sklearn.svm import LinearSVC

linear_svm = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LinearSVC(
        class_weight="balanced",
        max_iter=5000,
        random_state=42
    ))
])

linear_svm.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['dur', 'spkts', 'dpkts',
                                                   'sbytes', 'dbytes', 'rate',
                                                   'sttl', 'dttl', 'sload',
                                                   'dload', 'sloss', 'dloss',
                                                   'sinpkt', 'dinpkt', 'sjit',
                                                   'djit', 'swin', 'stcpb',
                                                   'dtcpb', 'dwin', 'tcprtt',
                                                   'synack', 'ackdat', 'smean',
                                                   'dmean', 'trans_depth',
                                                   'response_body_len',
                                                   'ct_srv_src', 'ct_state_ttl',
                                                   'ct_dst_ltm', ...]),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['proto', 'service',
                                                   'state'])])),
                ('classifier',
                 LinearSVC(class_weight='balanced', max_iter=5000,
                           random_state=42))])

In [24]:
svm_pred = linear_svm.predict(X_test)

print(classification_report(y_test, svm_pred))

              precision    recall  f1-score   support

           0       0.90      0.70      0.79     37000
           1       0.79      0.93      0.86     45332

    accuracy                           0.83     82332
   macro avg       0.85      0.82      0.82     82332
weighted avg       0.84      0.83      0.83     82332



In [25]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

svm_accuracy = accuracy_score(y_test, svm_pred)
svm_precision = precision_score(y_test, svm_pred)
svm_recall = recall_score(y_test, svm_pred)
svm_f1 = f1_score(y_test, svm_pred)

# LinearSVC has no predict_proba(), so ROC/PR AUC use decision scores
svm_scores = linear_svm.decision_function(X_test)

svm_roc_auc = roc_auc_score(y_test, svm_scores)
svm_pr_auc = average_precision_score(y_test, svm_scores)

print("===== NETSHIELD AI - LINEAR SVM =====")
print(f"Accuracy : {svm_accuracy:.4f}")
print(f"Precision: {svm_precision:.4f}")
print(f"Recall   : {svm_recall:.4f}")
print(f"F1 Score : {svm_f1:.4f}")
print(f"ROC-AUC  : {svm_roc_auc:.4f}")
print(f"PR-AUC   : {svm_pr_auc:.4f}")

print("\n===== CONFUSION MATRIX =====")
print(confusion_matrix(y_test, svm_pred))

===== NETSHIELD AI - LINEAR SVM =====
Accuracy : 0.8300
Precision: 0.7940
Recall   : 0.9334
F1 Score : 0.8581
ROC-AUC  : 0.9540
PR-AUC   : 0.9658

===== CONFUSION MATRIX =====
[[26024 10976]
 [ 3021 42311]]


In [26]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Linear SVM"],
    "Accuracy": [
        accuracy,
        svm_accuracy
    ],
    "Precision": [
        precision,
        svm_precision
    ],
    "Recall": [
        recall,
        svm_recall
    ],
    "F1": [
        f1,
        svm_f1
    ],
    "ROC-AUC": [
        roc_auc,
        svm_roc_auc
    ],
    "PR-AUC": [
        pr_auc,
        svm_pr_auc
    ]
})

results

,Model,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
0,Logistic Regression,0.835495,0.801903,0.931285,0.861765,0.956055,0.966752
1,Linear SVM,0.829993,0.794021,0.933358,0.858070,0.953978,0.965787


In [27]:
import joblib

linear_svm_path = f"{BASE_PATH}/models/linear_svm.pkl"

joblib.dump(linear_svm, linear_svm_path)

print("Saved:", linear_svm_path)

Saved: /content/drive/MyDrive/NetShield-AI/models/linear_svm.pkl


In [28]:
from sklearn.model_selection import train_test_split

X_rbf, _, y_rbf, _ = train_test_split(
    X_train,
    y_train,
    train_size=20000,
    stratify=y_train,
    random_state=42
)

print("RBF training sample:", X_rbf.shape)
print("Class distribution:")
print(y_rbf.value_counts())

RBF training sample: (20000, 42)
Class distribution:
label
1    13612
0     6388
Name: count, dtype: int64


In [29]:
from sklearn.svm import SVC

rbf_svm = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", SVC(
        kernel="rbf",
        class_weight="balanced"
    ))
])

In [30]:
rbf_svm.fit(X_rbf, y_rbf)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['dur', 'spkts', 'dpkts',
                                                   'sbytes', 'dbytes', 'rate',
                                                   'sttl', 'dttl', 'sload',
                                                   'dload', 'sloss', 'dloss',
                                                   'sinpkt', 'dinpkt', 'sjit',
                                                   'djit', 'swin', 'stcpb',
                                                   'dtcpb', 'dwin', 'tcprtt',
                                                   'synack', 'ackdat', 'smean',
                                                   'dmean', 'trans_depth',
                                                   'response_body_len',
                                                   'ct_srv_src', 'ct_state_ttl',
                                                   'ct_dst_ltm', ...]),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['proto', 'service',
                                                   'state'])])),
                ('classifier', SVC(class_weight='balanced'))])

In [31]:
rbf_pred = rbf_svm.predict(X_test)

print(classification_report(y_test, rbf_pred))

              precision    recall  f1-score   support

           0       0.93      0.72      0.81     37000
           1       0.81      0.96      0.88     45332

    accuracy                           0.85     82332
   macro avg       0.87      0.84      0.84     82332
weighted avg       0.86      0.85      0.85     82332



In [32]:
rbf_scores = rbf_svm.decision_function(X_test)

rbf_accuracy = accuracy_score(y_test, rbf_pred)
rbf_precision = precision_score(y_test, rbf_pred)
rbf_recall = recall_score(y_test, rbf_pred)
rbf_f1 = f1_score(y_test, rbf_pred)

rbf_roc_auc = roc_auc_score(y_test, rbf_scores)
rbf_pr_auc = average_precision_score(y_test, rbf_scores)

print("Accuracy :", rbf_accuracy)
print("Precision:", rbf_precision)
print("Recall   :", rbf_recall)
print("F1       :", rbf_f1)
print("ROC-AUC  :", rbf_roc_auc)
print("PR-AUC   :", rbf_pr_auc)

Accuracy : 0.8508963707914298
Precision: 0.8073375729852356
Recall   : 0.9577561104738375
F1       : 0.8761376248612652
ROC-AUC  : 0.9559410877943151
PR-AUC   : 0.9617309007084467


In [33]:
import joblib

rbf_path = f"{BASE_PATH}/models/rbf_svm.pkl"

joblib.dump(rbf_svm, rbf_path)

print("RBF model saved to:", rbf_path)

RBF model saved to: /content/drive/MyDrive/NetShield-AI/models/rbf_svm.pkl


In [34]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Linear SVM",
        "RBF SVM"
    ],
    "Accuracy": [
        accuracy,
        svm_accuracy,
        rbf_accuracy
    ],
    "Precision": [
        precision,
        svm_precision,
        rbf_precision
    ],
    "Recall": [
        recall,
        svm_recall,
        rbf_recall
    ],
    "F1": [
        f1,
        svm_f1,
        rbf_f1
    ],
    "ROC-AUC": [
        roc_auc,
        svm_roc_auc,
        rbf_roc_auc
    ],
    "PR-AUC": [
        pr_auc,
        svm_pr_auc,
        rbf_pr_auc
    ]
})

results

,Model,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
0,Logistic Regression,0.835495,0.801903,0.931285,0.861765,0.956055,0.966752
1,Linear SVM,0.829993,0.794021,0.933358,0.858070,0.953978,0.965787
2,RBF SVM,0.850896,0.807338,0.957756,0.876138,0.955941,0.961731


In [35]:
X_rbf
y_rbf

,label
56491,1
28360,0
163067,1
32454,0
94429,1
...,...
90510,1
52194,1
87190,1
164784,1


In [36]:
from sklearn.svm import SVC

poly_svm = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", SVC(
        kernel="poly",
        degree=3,
        C=1.0,
        gamma="scale",
        class_weight="balanced"
    ))
])

In [37]:
poly_svm.fit(X_rbf, y_rbf)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['dur', 'spkts', 'dpkts',
                                                   'sbytes', 'dbytes', 'rate',
                                                   'sttl', 'dttl', 'sload',
                                                   'dload', 'sloss', 'dloss',
                                                   'sinpkt', 'dinpkt', 'sjit',
                                                   'djit', 'swin', 'stcpb',
                                                   'dtcpb', 'dwin', 'tcprtt',
                                                   'synack', 'ackdat', 'smean',
                                                   'dmean', 'trans_depth',
                                                   'response_body_len',
                                                   'ct_srv_src', 'ct_state_ttl',
                                                   'ct_dst_ltm', ...]),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['proto', 'service',
                                                   'state'])])),
                ('classifier', SVC(class_weight='balanced', kernel='poly'))])

In [38]:
poly_pred = poly_svm.predict(X_test)

print(classification_report(y_test, poly_pred))

              precision    recall  f1-score   support

           0       0.96      0.67      0.79     37000
           1       0.78      0.98      0.87     45332

    accuracy                           0.84     82332
   macro avg       0.87      0.82      0.83     82332
weighted avg       0.86      0.84      0.83     82332



In [39]:
poly_scores = poly_svm.decision_function(X_test)

poly_accuracy = accuracy_score(y_test, poly_pred)
poly_precision = precision_score(y_test, poly_pred)
poly_recall = recall_score(y_test, poly_pred)
poly_f1 = f1_score(y_test, poly_pred)

poly_roc_auc = roc_auc_score(y_test, poly_scores)
poly_pr_auc = average_precision_score(y_test, poly_scores)

print("===== POLYNOMIAL SVM =====")
print(f"Accuracy : {poly_accuracy:.4f}")
print(f"Precision: {poly_precision:.4f}")
print(f"Recall   : {poly_recall:.4f}")
print(f"F1       : {poly_f1:.4f}")
print(f"ROC-AUC  : {poly_roc_auc:.4f}")
print(f"PR-AUC   : {poly_pr_auc:.4f}")

===== POLYNOMIAL SVM =====
Accuracy : 0.8392
Precision: 0.7847
Recall   : 0.9757
F1       : 0.8698
ROC-AUC  : 0.9396
PR-AUC   : 0.9189


In [40]:
import joblib

poly_path = f"{BASE_PATH}/models/polynomial_svm.pkl"

joblib.dump(poly_svm, poly_path)

print("Polynomial SVM saved to:", poly_path)

Polynomial SVM saved to: /content/drive/MyDrive/NetShield-AI/models/polynomial_svm.pkl


In [41]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Linear SVM",
        "RBF SVM",
        "Polynomial SVM"
    ],
    "Accuracy": [
        accuracy,
        svm_accuracy,
        rbf_accuracy,
        poly_accuracy
    ],
    "Precision": [
        precision,
        svm_precision,
        rbf_precision,
        poly_precision
    ],
    "Recall": [
        recall,
        svm_recall,
        rbf_recall,
        poly_recall
    ],
    "F1": [
        f1,
        svm_f1,
        rbf_f1,
        poly_f1
    ],
    "ROC-AUC": [
        roc_auc,
        svm_roc_auc,
        rbf_roc_auc,
        poly_roc_auc
    ],
    "PR-AUC": [
        pr_auc,
        svm_pr_auc,
        rbf_pr_auc,
        poly_pr_auc
    ]
})

results

,Model,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
0,Logistic Regression,0.835495,0.801903,0.931285,0.861765,0.956055,0.966752
1,Linear SVM,0.829993,0.794021,0.933358,0.858070,0.953978,0.965787
2,RBF SVM,0.850896,0.807338,0.957756,0.876138,0.955941,0.961731
3,Polynomial SVM,0.839188,0.784655,0.975713,0.869816,0.939649,0.918909


In [42]:
X_train_multi = train.drop(
    columns=["id", "label", "attack_cat"]
)

y_train_multi = train["attack_cat"]

X_test_multi = test.drop(
    columns=["id", "label", "attack_cat"]
)

y_test_multi = test["attack_cat"]

print("Training:", X_train_multi.shape)
print("Testing :", X_test_multi.shape)

print("\nClasses:")
print(y_train_multi.value_counts())

Training: (175341, 42)
Testing : (82332, 42)

Classes:
attack_cat
Normal            56000
Generic           40000
Exploits          33393
Fuzzers           18184
DoS               12264
Reconnaissance    10491
Analysis           2000
Backdoor           1746
Shellcode          1133
Worms               130
Name: count, dtype: int64


In [43]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

multiclass_logistic = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

In [44]:
multiclass_logistic.fit(
    X_train_multi,
    y_train_multi
)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['dur', 'spkts', 'dpkts',
                                                   'sbytes', 'dbytes', 'rate',
                                                   'sttl', 'dttl', 'sload',
                                                   'dload', 'sloss', 'dloss',
                                                   'sinpkt', 'dinpkt', 'sjit',
                                                   'djit', 'swin', 'stcpb',
                                                   'dtcpb', 'dwin', 'tcprtt',
                                                   'synack', 'ackdat', 'smean',
                                                   'dmean', 'trans_depth',
                                                   'response_body_len',
                                                   'ct_srv_src', 'ct_state_ttl',
                                                   'ct_dst_ltm', ...]),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['proto', 'service',
                                                   'state'])])),
                ('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

In [45]:
multi_pred = multiclass_logistic.predict(X_test_multi)

print(multi_pred[:20])

['Shellcode' 'Worms' 'Fuzzers' 'Fuzzers' 'Fuzzers' 'Fuzzers' 'Fuzzers'
 'Fuzzers' 'Normal' 'Normal' 'Normal' 'Normal' 'Fuzzers' 'Fuzzers' 'Worms'
 'Shellcode' 'Shellcode' 'Worms' 'Shellcode' 'Worms']


In [46]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

multi_accuracy = accuracy_score(
    y_test_multi,
    multi_pred
)

multi_macro_f1 = f1_score(
    y_test_multi,
    multi_pred,
    average="macro"
)

multi_weighted_f1 = f1_score(
    y_test_multi,
    multi_pred,
    average="weighted"
)

print("Multiclass Accuracy :", multi_accuracy)
print("Multiclass Macro-F1 :", multi_macro_f1)
print("Multiclass Weighted-F1:", multi_weighted_f1)

print("\n===== CLASSIFICATION REPORT =====")
print(
    classification_report(
        y_test_multi,
        multi_pred
    )
)

Multiclass Accuracy : 0.6205242190156925
Multiclass Macro-F1 : 0.35169309909341057
Multiclass Weighted-F1: 0.6848763282543374

===== CLASSIFICATION REPORT =====
                precision    recall  f1-score   support

      Analysis       0.06      0.57      0.11       677
      Backdoor       0.03      0.16      0.05       583
           DoS       0.41      0.14      0.21      4089
      Exploits       0.80      0.45      0.58     11132
       Fuzzers       0.28      0.64      0.39      6062
       Generic       1.00      0.96      0.98     18871
        Normal       0.98      0.58      0.73     37000
Reconnaissance       0.33      0.33      0.33      3496
     Shellcode       0.06      0.82      0.11       378
         Worms       0.02      0.82      0.03        44

      accuracy                           0.62     82332
     macro avg       0.40      0.55      0.35     82332
  weighted avg       0.83      0.62      0.68     82332



In [47]:
cm = confusion_matrix(
    y_test_multi,
    multi_pred,
    labels=multiclass_logistic.classes_
)

print(cm)

[[  389   145    66     0    18     0    58     1     0     0]
 [  368    95    44     9    27     0     1    27     8     4]
 [ 1172  1194   589   546   146    23    45    90   123   161]
 [ 1546  1230   538  5037   684    31   246   155   494  1171]
 [  814   301   102    36  3886     3    15   329   461   115]
 [   19    25    19   287   142 18168    12    29    52   118]
 [ 1955    16     9   356  9051     1 21430  1681  2409    92]
 [   93   159    73    11   137     7     0  1148  1434   434]
 [    0     0     0     0    22     0     0    41   311     4]
 [    0     0     0     0     3     0     0     1     4    36]]


In [48]:
print("Class order:")
print(multiclass_logistic.classes_)

Class order:
['Analysis' 'Backdoor' 'DoS' 'Exploits' 'Fuzzers' 'Generic' 'Normal'
 'Reconnaissance' 'Shellcode' 'Worms']


In [49]:
print(classification_report(
    y_test_multi,
    multi_pred,
    labels=multiclass_logistic.classes_,
    digits=4
))

                precision    recall  f1-score   support

      Analysis     0.0612    0.5746    0.1106       677
      Backdoor     0.0300    0.1630    0.0507       583
           DoS     0.4090    0.1440    0.2131      4089
      Exploits     0.8018    0.4525    0.5785     11132
       Fuzzers     0.2753    0.6410    0.3852      6062
       Generic     0.9964    0.9627    0.9793     18871
        Normal     0.9827    0.5792    0.7288     37000
Reconnaissance     0.3278    0.3284    0.3281      3496
     Shellcode     0.0587    0.8228    0.1096       378
         Worms     0.0169    0.8182    0.0330        44

      accuracy                         0.6205     82332
     macro avg     0.3960    0.5486    0.3517     82332
  weighted avg     0.8339    0.6205    0.6849     82332



In [50]:
multi_accuracy = accuracy_score(y_test_multi, multi_pred)
multi_macro_f1 = f1_score(
    y_test_multi,
    multi_pred,
    average="macro"
)
multi_weighted_f1 = f1_score(
    y_test_multi,
    multi_pred,
    average="weighted"
)

print("===== MULTICLASS LOGISTIC REGRESSION =====")
print(f"Accuracy      : {multi_accuracy:.4f}")
print(f"Macro F1      : {multi_macro_f1:.4f}")
print(f"Weighted F1   : {multi_weighted_f1:.4f}")

===== MULTICLASS LOGISTIC REGRESSION =====
Accuracy      : 0.6205
Macro F1      : 0.3517
Weighted F1   : 0.6849


In [51]:
import joblib

multi_path = f"{BASE_PATH}/models/multiclass_logistic.pkl"

joblib.dump(multiclass_logistic, multi_path)

print("Saved:", multi_path)

Saved: /content/drive/MyDrive/NetShield-AI/models/multiclass_logistic.pkl


In [52]:
from sklearn.svm import LinearSVC

multiclass_svm = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LinearSVC(
        class_weight="balanced",
        max_iter=5000,
        random_state=42
    ))
])

In [53]:
multiclass_svm.fit(
    X_train_multi,
    y_train_multi
)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['dur', 'spkts', 'dpkts',
                                                   'sbytes', 'dbytes', 'rate',
                                                   'sttl', 'dttl', 'sload',
                                                   'dload', 'sloss', 'dloss',
                                                   'sinpkt', 'dinpkt', 'sjit',
                                                   'djit', 'swin', 'stcpb',
                                                   'dtcpb', 'dwin', 'tcprtt',
                                                   'synack', 'ackdat', 'smean',
                                                   'dmean', 'trans_depth',
                                                   'response_body_len',
                                                   'ct_srv_src', 'ct_state_ttl',
                                                   'ct_dst_ltm', ...]),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['proto', 'service',
                                                   'state'])])),
                ('classifier',
                 LinearSVC(class_weight='balanced', max_iter=5000,
                           random_state=42))])

In [54]:
multi_svm_pred = multiclass_svm.predict(X_test_multi)

In [55]:
print(classification_report(
    y_test_multi,
    multi_svm_pred,
    digits=4
))

                precision    recall  f1-score   support

      Analysis     0.0898    0.5938    0.1560       677
      Backdoor     0.0347    0.0480    0.0403       583
           DoS     0.4207    0.4696    0.4438      4089
      Exploits     0.6964    0.4860    0.5725     11132
       Fuzzers     0.2659    0.6381    0.3754      6062
       Generic     0.9940    0.9624    0.9780     18871
        Normal     0.9915    0.5795    0.7315     37000
Reconnaissance     0.4378    0.6719    0.5302      3496
     Shellcode     0.0785    0.5979    0.1387       378
         Worms     0.0178    0.8182    0.0348        44

      accuracy                         0.6540     82332
     macro avg     0.4027    0.5865    0.4001     82332
  weighted avg     0.8280    0.6540    0.7047     82332



In [56]:
multi_svm_macro_f1 = f1_score(
    y_test_multi,
    multi_svm_pred,
    average="macro"
)

print("Multiclass Linear SVM Macro-F1:",
      multi_svm_macro_f1)

Multiclass Linear SVM Macro-F1: 0.4001064659343775


In [57]:
import joblib

multi_svm_path = f"{BASE_PATH}/models/multiclass_linear_svm.pkl"

joblib.dump(multiclass_svm, multi_svm_path)

print("Saved:", multi_svm_path)

Saved: /content/drive/MyDrive/NetShield-AI/models/multiclass_linear_svm.pkl


In [58]:
multi_logistic_path = f"{BASE_PATH}/models/multiclass_logistic.pkl"

joblib.dump(multiclass_logistic, multi_logistic_path)

print("Saved:", multi_logistic_path)

Saved: /content/drive/MyDrive/NetShield-AI/models/multiclass_logistic.pkl


In [59]:
multi_results = pd.DataFrame({
    "Model": [
        "Multiclass Logistic Regression",
        "Multiclass Linear SVM"
    ],
    "Accuracy": [
        multi_accuracy,
        accuracy_score(y_test_multi, multi_svm_pred)
    ],
    "Macro-F1": [
        multi_macro_f1,
        multi_svm_macro_f1
    ],
    "Weighted-F1": [
        multi_weighted_f1,
        f1_score(
            y_test_multi,
            multi_svm_pred,
            average="weighted"
        )
    ]
})

multi_results

,Model,Accuracy,Macro-F1,Weighted-F1
0,Multiclass Logistic Regression,0.620524,0.351693,0.684876
1,Multiclass Linear SVM,0.653986,0.400106,0.704715


In [60]:
X_reg_train = train.drop(
    columns=["id", "label", "attack_cat", "sbytes", "dbytes"]
)

X_reg_test = test.drop(
    columns=["id", "label", "attack_cat", "sbytes", "dbytes"]
)

y_reg_train = train["sbytes"] + train["dbytes"]
y_reg_test = test["sbytes"] + test["dbytes"]

print("X_reg_train:", X_reg_train.shape)
print("y_reg_train:", y_reg_train.shape)

X_reg_train: (175341, 40)
y_reg_train: (175341,)


In [61]:
reg_categorical_cols = ["proto", "service", "state"]

reg_numerical_cols = [
    col for col in X_reg_train.columns
    if col not in reg_categorical_cols
]

reg_preprocessor = ColumnTransformer([
    (
        "num",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]),
        reg_numerical_cols
    ),
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]),
        reg_categorical_cols
    )
])

In [62]:
from sklearn.linear_model import LinearRegression

linear_regression = Pipeline([
    ("preprocessor", reg_preprocessor),
    ("regressor", LinearRegression())
])

In [63]:
linear_regression.fit(
    X_reg_train,
    y_reg_train
)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['dur', 'spkts', 'dpkts',
                                                   'rate', 'sttl', 'dttl',
                                                   'sload', 'dload', 'sloss',
                                                   'dloss', 'sinpkt', 'dinpkt',
                                                   'sjit', 'djit', 'swin',
                                                   'stcpb', 'dtcpb', 'dwin',
                                                   'tcprtt', 'synack', 'ackdat',
                                                   'smean', 'dmean',
                                                   'trans_depth',
                                                   'response_body_len',
                                                   'ct_srv_src', 'ct_state_ttl',
                                                   'ct_dst_ltm',
                                                   'ct_src_dport_ltm',
                                                   'ct_dst_sport_ltm', ...]),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['proto', 'service',
                                                   'state'])])),
                ('regressor', LinearRegression())])

In [64]:
reg_pred = linear_regression.predict(X_reg_test)

In [65]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    median_absolute_error
)

mae = mean_absolute_error(y_reg_test, reg_pred)
rmse = np.sqrt(mean_squared_error(y_reg_test, reg_pred))
r2 = r2_score(y_reg_test, reg_pred)
medae = median_absolute_error(y_reg_test, reg_pred)

print("===== LINEAR REGRESSION =====")
print(f"MAE     : {mae:.4f}")
print(f"RMSE    : {rmse:.4f}")
print(f"R²      : {r2:.4f}")
print(f"MedAE   : {medae:.4f}")

===== LINEAR REGRESSION =====
MAE     : 5274.3892
RMSE    : 15610.8182
R²      : 0.9954
MedAE   : 3150.9970


In [66]:
import joblib

linear_reg_path = f"{BASE_PATH}/models/linear_regression.pkl"

joblib.dump(linear_regression, linear_reg_path)

print("Saved:", linear_reg_path)

Saved: /content/drive/MyDrive/NetShield-AI/models/linear_regression.pkl


In [67]:
from sklearn.linear_model import Ridge

ridge_regression = Pipeline([
    ("preprocessor", reg_preprocessor),
    ("regressor", Ridge(alpha=1.0))
])

In [68]:
ridge_regression.fit(
    X_reg_train,
    y_reg_train
)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['dur', 'spkts', 'dpkts',
                                                   'rate', 'sttl', 'dttl',
                                                   'sload', 'dload', 'sloss',
                                                   'dloss', 'sinpkt', 'dinpkt',
                                                   'sjit', 'djit', 'swin',
                                                   'stcpb', 'dtcpb', 'dwin',
                                                   'tcprtt', 'synack', 'ackdat',
                                                   'smean', 'dmean',
                                                   'trans_depth',
                                                   'response_body_len',
                                                   'ct_srv_src', 'ct_state_ttl',
                                                   'ct_dst_ltm',
                                                   'ct_src_dport_ltm',
                                                   'ct_dst_sport_ltm', ...]),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['proto', 'service',
                                                   'state'])])),
                ('regressor', Ridge())])

In [69]:
ridge_pred = ridge_regression.predict(X_reg_test)

In [70]:
ridge_mae = mean_absolute_error(y_reg_test, ridge_pred)
ridge_rmse = np.sqrt(mean_squared_error(y_reg_test, ridge_pred))
ridge_r2 = r2_score(y_reg_test, ridge_pred)
ridge_medae = median_absolute_error(y_reg_test, ridge_pred)

print("===== RIDGE REGRESSION =====")
print(f"MAE   : {ridge_mae:.4f}")
print(f"RMSE  : {ridge_rmse:.4f}")
print(f"R²    : {ridge_r2:.4f}")
print(f"MedAE : {ridge_medae:.4f}")

===== RIDGE REGRESSION =====
MAE   : 5102.5143
RMSE  : 15571.1567
R²    : 0.9954
MedAE : 2723.1081


In [71]:
ridge_path = f"{BASE_PATH}/models/ridge_regression.pkl"

joblib.dump(ridge_regression, ridge_path)

print("Saved:", ridge_path)

Saved: /content/drive/MyDrive/NetShield-AI/models/ridge_regression.pkl


In [72]:
from sklearn.linear_model import HuberRegressor

huber_regression = Pipeline([
    ("preprocessor", reg_preprocessor),
    ("regressor", HuberRegressor(
        max_iter=500
    ))
])

In [73]:
huber_regression.fit(
    X_reg_train,
    y_reg_train
)

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_huber.py:343: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['dur', 'spkts', 'dpkts',
                                                   'rate', 'sttl', 'dttl',
                                                   'sload', 'dload', 'sloss',
                                                   'dloss', 'sinpkt', 'dinpkt',
                                                   'sjit', 'djit', 'swin',
                                                   'stcpb', 'dtcpb', 'dwin',
                                                   'tcprtt', 'synack', 'ackdat',
                                                   'smean', 'dmean',
                                                   'trans_depth',
                                                   'response_body_len',
                                                   'ct_srv_src', 'ct_state_ttl',
                                                   'ct_dst_ltm',
                                                   'ct_src_dport_ltm',
                                                   'ct_dst_sport_ltm', ...]),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['proto', 'service',
                                                   'state'])])),
                ('regressor', HuberRegressor(max_iter=500))])

In [74]:
huber_pred = huber_regression.predict(X_reg_test)

In [75]:
huber_mae = mean_absolute_error(y_reg_test, huber_pred)
huber_rmse = np.sqrt(mean_squared_error(y_reg_test, huber_pred))
huber_r2 = r2_score(y_reg_test, huber_pred)
huber_medae = median_absolute_error(y_reg_test, huber_pred)

print("===== HUBER REGRESSION =====")
print(f"MAE   : {huber_mae:.4f}")
print(f"RMSE  : {huber_rmse:.4f}")
print(f"R²    : {huber_r2:.4f}")
print(f"MedAE : {huber_medae:.4f}")

===== HUBER REGRESSION =====
MAE   : 3847.3106
RMSE  : 22762.7652
R²    : 0.9902
MedAE : 164.5951


In [76]:
huber_path = f"{BASE_PATH}/models/huber_regression.pkl"

joblib.dump(huber_regression, huber_path)

print("Saved:", huber_path)

Saved: /content/drive/MyDrive/NetShield-AI/models/huber_regression.pkl


In [77]:
regression_results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Ridge Regression",
        "Huber Regression"
    ],
    "MAE": [
        mae,
        ridge_mae,
        huber_mae
    ],
    "RMSE": [
        rmse,
        ridge_rmse,
        huber_rmse
    ],
    "R²": [
        r2,
        ridge_r2,
        huber_r2
    ],
    "Median AE": [
        medae,
        ridge_medae,
        huber_medae
    ]
})

regression_results

,Model,MAE,RMSE,R²,Median AE
0,Linear Regression,5274.389203,15610.818194,0.995395,3150.996965
1,Ridge Regression,5102.514340,15571.156724,0.995419,2723.108092
2,Huber Regression,3847.310567,22762.765249,0.990210,164.595131


In [78]:
from sklearn.model_selection import train_test_split

X_svr, _, y_svr, _ = train_test_split(
    X_reg_train,
    y_reg_train,
    train_size=20000,
    random_state=42
)

print(X_svr.shape)
print(y_svr.shape)

(20000, 40)
(20000,)


In [79]:
from sklearn.svm import SVR

svr_model = Pipeline([
    ("preprocessor", reg_preprocessor),
    ("regressor", SVR(
        kernel="rbf",
        C=10.0,
        epsilon=0.1
    ))
])

In [80]:
svr_model.fit(X_svr, y_svr)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['dur', 'spkts', 'dpkts',
                                                   'rate', 'sttl', 'dttl',
                                                   'sload', 'dload', 'sloss',
                                                   'dloss', 'sinpkt', 'dinpkt',
                                                   'sjit', 'djit', 'swin',
                                                   'stcpb', 'dtcpb', 'dwin',
                                                   'tcprtt', 'synack', 'ackdat',
                                                   'smean', 'dmean',
                                                   'trans_depth',
                                                   'response_body_len',
                                                   'ct_srv_src', 'ct_state_ttl',
                                                   'ct_dst_ltm',
                                                   'ct_src_dport_ltm',
                                                   'ct_dst_sport_ltm', ...]),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['proto', 'service',
                                                   'state'])])),
                ('regressor', SVR(C=10.0))])

In [81]:
svr_pred = svr_model.predict(X_reg_test)

In [82]:
svr_mae = mean_absolute_error(y_reg_test, svr_pred)
svr_rmse = np.sqrt(mean_squared_error(y_reg_test, svr_pred))
svr_r2 = r2_score(y_reg_test, svr_pred)
svr_medae = median_absolute_error(y_reg_test, svr_pred)

print("===== SVR =====")
print(f"MAE   : {svr_mae:.4f}")
print(f"RMSE  : {svr_rmse:.4f}")
print(f"R²    : {svr_r2:.4f}")
print(f"MedAE : {svr_medae:.4f}")

===== SVR =====
MAE   : 19557.0044
RMSE  : 230580.6537
R²    : -0.0046
MedAE : 230.5847


In [83]:
svr_path = f"{BASE_PATH}/models/svr.pkl"
joblib.dump(svr_model, svr_path)

print("Saved:", svr_path)

Saved: /content/drive/MyDrive/NetShield-AI/models/svr.pkl


In [84]:
regression_results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Ridge Regression",
        "Huber Regression",
        "SVR"
    ],
    "MAE": [
        mae,
        ridge_mae,
        huber_mae,
        svr_mae
    ],
    "RMSE": [
        rmse,
        ridge_rmse,
        huber_rmse,
        svr_rmse
    ],
    "R²": [
        r2,
        ridge_r2,
        huber_r2,
        svr_r2
    ],
    "Median AE": [
        medae,
        ridge_medae,
        huber_medae,
        svr_medae
    ]
})

regression_results

,Model,MAE,RMSE,R²,Median AE
0,Linear Regression,5274.389203,15610.818194,0.995395,3150.996965
1,Ridge Regression,5102.514340,15571.156724,0.995419,2723.108092
2,Huber Regression,3847.310567,22762.765249,0.990210,164.595131
3,SVR,19557.004428,230580.653739,-0.004564,230.584712


In [85]:
from sklearn.linear_model import BayesianRidge

bayesian_linear = Pipeline([
    ("preprocessor", reg_preprocessor),
    ("regressor", BayesianRidge())
])

In [86]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

bayesian_reg_preprocessor = ColumnTransformer([
    (
        "num",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]),
        reg_numerical_cols
    ),
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ))
        ]),
        reg_categorical_cols
    )
])

In [87]:
from sklearn.linear_model import BayesianRidge

bayesian_linear = Pipeline([
    ("preprocessor", bayesian_reg_preprocessor),
    ("regressor", BayesianRidge())
])

In [88]:
bayesian_linear.fit(
    X_reg_train,
    y_reg_train
)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['dur', 'spkts', 'dpkts',
                                                   'rate', 'sttl', 'dttl',
                                                   'sload', 'dload', 'sloss',
                                                   'dloss', 'sinpkt', 'dinpkt',
                                                   'sjit', 'djit', 'swin',
                                                   'stcpb', 'dtcpb', 'dwin',
                                                   'tcprtt', 'synack', 'ackdat',
                                                   'smean', 'dmean',
                                                   'trans_depth',
                                                   'response_body_len',
                                                   'ct_srv_src', 'ct_state_ttl',
                                                   'ct_dst_ltm',
                                                   'ct_src_dport_ltm',
                                                   'ct_dst_sport_ltm', ...]),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['proto', 'service',
                                                   'state'])])),
                ('regressor', BayesianRidge())])

In [89]:
bayesian_pred = bayesian_linear.predict(X_reg_test)

print(bayesian_pred[:10])

[ 660.29993289 2890.09361695 1023.57618983 1141.51067895 4121.60896055
   45.5820501  2642.66804116 3217.18978423  192.88311673  192.88395832]


In [90]:
bayesian_mae = mean_absolute_error(
    y_reg_test,
    bayesian_pred
)

bayesian_rmse = np.sqrt(
    mean_squared_error(y_reg_test, bayesian_pred)
)

bayesian_r2 = r2_score(
    y_reg_test,
    bayesian_pred
)

bayesian_medae = median_absolute_error(
    y_reg_test,
    bayesian_pred
)

print("===== BAYESIAN LINEAR REGRESSION =====")
print(f"MAE   : {bayesian_mae:.4f}")
print(f"RMSE  : {bayesian_rmse:.4f}")
print(f"R²    : {bayesian_r2:.4f}")
print(f"MedAE : {bayesian_medae:.4f}")

===== BAYESIAN LINEAR REGRESSION =====
MAE   : 5273.8416
RMSE  : 15610.9363
R²    : 0.9954
MedAE : 3150.7310


In [91]:
bayesian_linear_path = f"{BASE_PATH}/models/bayesian_linear.pkl"

joblib.dump(
    bayesian_linear,
    bayesian_linear_path
)

print("Saved:", bayesian_linear_path)

Saved: /content/drive/MyDrive/NetShield-AI/models/bayesian_linear.pkl


In [92]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

bayes_preprocessor = ColumnTransformer([
    (
        "num",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]),
        numerical_cols
    ),
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ))
        ]),
        categorical_cols
    )
])

In [93]:
X_train_bayes = bayes_preprocessor.fit_transform(X_train)
X_test_bayes = bayes_preprocessor.transform(X_test)

print("Bayesian train matrix:", X_train_bayes.shape)
print("Bayesian test matrix :", X_test_bayes.shape)

Bayesian train matrix: (175341, 194)
Bayesian test matrix : (82332, 194)


In [94]:
X_train_bayes = X_train_bayes.astype(np.float64)
X_test_bayes = X_test_bayes.astype(np.float64)

y_train_bayes = y_train.to_numpy()
y_test_bayes = y_test.to_numpy()

In [95]:
X_train_bayes = np.column_stack([
    np.ones(X_train_bayes.shape[0]),
    X_train_bayes
])

X_test_bayes = np.column_stack([
    np.ones(X_test_bayes.shape[0]),
    X_test_bayes
])

print(X_train_bayes.shape)

(175341, 195)


In [96]:
from scipy.special import expit
from scipy.optimize import minimize

n_features = X_train_bayes.shape[1]

m0 = np.zeros(n_features)

prior_variance = 10.0
prior_precision = 1.0 / prior_variance

S0_inv = prior_precision * np.eye(n_features)

In [97]:
def negative_log_posterior(w, X, y):
    z = X @ w

    # Stable negative log-likelihood
    nll = np.sum(
        np.logaddexp(0, z) - y * z
    )

    # Gaussian prior
    diff = w - m0
    prior = 0.5 * diff @ S0_inv @ diff

    return nll + prior

In [98]:
def gradient_negative_log_posterior(w, X, y):
    z = X @ w
    p = expit(z)

    gradient = X.T @ (p - y)

    diff = w - m0
    gradient += S0_inv @ diff

    return gradient

In [99]:
w_initial = np.zeros(n_features)

In [100]:
result = minimize(
    negative_log_posterior,
    w_initial,
    args=(X_train_bayes, y_train_bayes),
    jac=gradient_negative_log_posterior,
    method="L-BFGS-B",
    options={
        "maxiter": 1000,
        "disp": True
    }
)

print("Optimization success:", result.success)
print("Message:", result.message)

/tmp/ipykernel_7028/244209597.py:1: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  result = minimize(


Optimization success: False
Message: STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT


In [101]:
# Get the fitted Logistic Regression coefficients
sklearn_logistic = logistic_model.named_steps["classifier"]

w_logistic = sklearn_logistic.coef_.ravel()
bias_logistic = sklearn_logistic.intercept_[0]

w_initial = np.concatenate([
    [bias_logistic],
    w_logistic
])

print("Initial weight vector:", w_initial.shape)

Initial weight vector: (195,)


In [102]:
result = minimize(
    negative_log_posterior,
    w_initial,
    args=(X_train_bayes, y_train_bayes),
    jac=gradient_negative_log_posterior,
    method="L-BFGS-B",
    options={
        "maxiter": 2000,
        "gtol": 1e-5,
        "ftol": 1e-9,
        "maxls": 50
    }
)

print("Optimization success:", result.success)
print("Message:", result.message)
print("Iterations:", result.nit)

Optimization success: True
Message: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
Iterations: 1324


In [103]:
w_map = result.x
print("Number of weights:", len(w_map))

Number of weights: 195


In [104]:
def calculate_hessian(w, X):
    z = X @ w
    p = expit(z)

    r = p * (1 - p)

    weighted_X = X * r[:, np.newaxis]

    H = X.T @ weighted_X + S0_inv

    return H

In [105]:
H = calculate_hessian(
    w_map,
    X_train_bayes
)

print("Hessian shape:", H.shape)

Hessian shape: (195, 195)


In [106]:
posterior_covariance = np.linalg.inv(H)

print(
    "Posterior covariance shape:",
    posterior_covariance.shape
)

Posterior covariance shape: (195, 195)


In [107]:
posterior_covariance = (
    posterior_covariance +
    posterior_covariance.T
) / 2

In [108]:
map_probability = expit(
    X_test_bayes @ w_map
)

print(map_probability[:10])

[8.72684553e-01 8.69792469e-01 8.58555427e-01 8.83536303e-01
 8.69173927e-01 9.10368079e-01 9.05825933e-01 8.28440880e-01
 2.02751062e-06 2.02751152e-06]


In [109]:
n_samples = 200

posterior_samples = np.random.multivariate_normal(
    mean=w_map,
    cov=posterior_covariance,
    size=n_samples
)

print(
    "Posterior samples:",
    posterior_samples.shape
)

Posterior samples: (200, 195)


In [110]:
all_probabilities = []

for w in posterior_samples:

    p = expit(
        X_test_bayes @ w
    )

    all_probabilities.append(p)

all_probabilities = np.array(
    all_probabilities
)

print(all_probabilities.shape)

(200, 82332)


In [111]:
bayesian_attack_probability = np.mean(
    all_probabilities,
    axis=0
)

In [112]:
lower_probability = np.percentile(
    all_probabilities,
    2.5,
    axis=0
)

upper_probability = np.percentile(
    all_probabilities,
    97.5,
    axis=0
)

In [113]:
for i in range(10):

    print(
        f"Flow {i}: "
        f"{bayesian_attack_probability[i]:.4f} "
        f"[{lower_probability[i]:.4f}, "
        f"{upper_probability[i]:.4f}]"
    )

Flow 0: 0.8726 [0.8626, 0.8803]
Flow 1: 0.8692 [0.8583, 0.8790]
Flow 2: 0.8581 [0.8475, 0.8684]
Flow 3: 0.8834 [0.8750, 0.8907]
Flow 4: 0.8686 [0.8563, 0.8792]
Flow 5: 0.9102 [0.9022, 0.9174]
Flow 6: 0.9053 [0.8939, 0.9150]
Flow 7: 0.8280 [0.8126, 0.8404]
Flow 8: 0.1517 [0.0000, 1.0000]
Flow 9: 0.1517 [0.0000, 1.0000]


In [114]:
bayesian_pred = (
    bayesian_attack_probability >= 0.5
).astype(int)

In [115]:
bayes_accuracy = accuracy_score(
    y_test_bayes,
    bayesian_pred
)

bayes_precision = precision_score(
    y_test_bayes,
    bayesian_pred
)

bayes_recall = recall_score(
    y_test_bayes,
    bayesian_pred
)

bayes_f1 = f1_score(
    y_test_bayes,
    bayesian_pred
)

bayes_roc_auc = roc_auc_score(
    y_test_bayes,
    bayesian_attack_probability
)

bayes_pr_auc = average_precision_score(
    y_test_bayes,
    bayesian_attack_probability
)

print("===== BAYESIAN LOGISTIC REGRESSION =====")
print(f"Accuracy : {bayes_accuracy:.4f}")
print(f"Precision: {bayes_precision:.4f}")
print(f"Recall   : {bayes_recall:.4f}")
print(f"F1       : {bayes_f1:.4f}")
print(f"ROC-AUC  : {bayes_roc_auc:.4f}")
print(f"PR-AUC   : {bayes_pr_auc:.4f}")

===== BAYESIAN LOGISTIC REGRESSION =====
Accuracy : 0.8119
Precision: 0.7553
Recall   : 0.9738
F1       : 0.8508
ROC-AUC  : 0.9561
PR-AUC   : 0.9663


In [116]:
bayesian_logistic_model = {
    "preprocessor": bayes_preprocessor,
    "weights_map": w_map,
    "posterior_covariance": posterior_covariance,
    "prior_mean": m0,
    "prior_variance": prior_variance,
    "threshold": 0.5
}

In [117]:
bayesian_logistic_path = (
    f"{BASE_PATH}/models/bayesian_logistic_laplace.pkl"
)

joblib.dump(
    bayesian_logistic_model,
    bayesian_logistic_path
)

print("Saved:", bayesian_logistic_path)

Saved: /content/drive/MyDrive/NetShield-AI/models/bayesian_logistic_laplace.pkl


In [118]:
# Use the already-fitted dense Bayesian preprocessing
X_kernel_train_full = bayes_preprocessor.transform(X_train)
X_kernel_test = bayes_preprocessor.transform(X_test)

y_kernel_train_full = y_train.to_numpy()
y_kernel_test = y_test.to_numpy()

print("Train:", X_kernel_train_full.shape)
print("Test :", X_kernel_test.shape)

Train: (175341, 194)
Test : (82332, 194)


In [119]:
from sklearn.model_selection import train_test_split

X_kernel_train, _, y_kernel_train, _ = train_test_split(
    X_kernel_train_full,
    y_kernel_train_full,
    train_size=10000,
    stratify=y_kernel_train_full,
    random_state=42
)

print("Kernel training shape:", X_kernel_train.shape)
print("Class distribution:")
print(pd.Series(y_kernel_train).value_counts())

Kernel training shape: (10000, 194)
Class distribution:
1    6806
0    3194
Name: count, dtype: int64


In [120]:
def laplacian_kernel(X, Y, gamma=0.05):
    distance = np.sum(
        np.abs(X[:, None, :] - Y[None, :, :]),
        axis=2
    )

    return np.exp(-gamma * distance)

In [121]:
def laplacian_kernel(X, Y):
    gamma = 0.05

    distance = np.sum(
        np.abs(X[:, None, :] - Y[None, :, :]),
        axis=2
    )

    return np.exp(-gamma * distance)

In [122]:
custom_svm = SVC(
    kernel=laplacian_kernel,
    class_weight="balanced",
    cache_size=2000
)

In [8]:
# ============================================
# NETSHIELD AI - REBUILD DATA VARIABLES
# ============================================

import pandas as pd
import numpy as np

# Base path
BASE_PATH = "/content/drive/MyDrive/NetShield-AI"

# Dataset paths
train_path = f"{BASE_PATH}/data/raw/UNSW_NB15_training-set.csv"
test_path = f"{BASE_PATH}/data/raw/UNSW_NB15_testing-set.csv"

# Load datasets
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

print("Training shape:", train.shape)
print("Testing shape :", test.shape)

# ============================================
# Binary classification data
# ============================================

X_train = train.drop(columns=["id", "label", "attack_cat"])
y_train = train["label"]

X_test = test.drop(columns=["id", "label", "attack_cat"])
y_test = test["label"]

# Replace infinite values
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

# ============================================
# Feature types
# ============================================

categorical_cols = ["proto", "service", "state"]

numerical_cols = [
    col for col in X_train.columns
    if col not in categorical_cols
]

print("Numerical columns:", len(numerical_cols))
print("Categorical columns:", len(categorical_cols))

Training shape: (175341, 45)
Testing shape : (82332, 45)
Numerical columns: 39
Categorical columns: 3


In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

bayes_preprocessor = ColumnTransformer([
    (
        "num",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]),
        numerical_cols
    ),
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ))
        ]),
        categorical_cols
    )
])

print("Preprocessor ready.")

Preprocessor ready.


In [10]:
X_kernel_train_full = bayes_preprocessor.fit_transform(X_train)
X_kernel_test = bayes_preprocessor.transform(X_test)

y_kernel_train_full = y_train.to_numpy()
y_kernel_test = y_test.to_numpy()

print("Kernel train:", X_kernel_train_full.shape)
print("Kernel test :", X_kernel_test.shape)

Kernel train: (175341, 194)
Kernel test : (82332, 194)


In [11]:
from sklearn.model_selection import train_test_split

X_kernel_train, _, y_kernel_train, _ = train_test_split(
    X_kernel_train_full,
    y_kernel_train_full,
    train_size=10000,
    stratify=y_kernel_train_full,
    random_state=42
)

print("Kernel training shape:", X_kernel_train.shape)
print(pd.Series(y_kernel_train).value_counts())

Kernel training shape: (10000, 194)
1    6806
0    3194
Name: count, dtype: int64


In [12]:
from sklearn.model_selection import train_test_split

X_kernel_train, _, y_kernel_train, _ = train_test_split(
    X_kernel_train_full,
    y_kernel_train_full,
    train_size=5000,
    stratify=y_kernel_train_full,
    random_state=42
)

print("Kernel training shape:", X_kernel_train.shape)
print(pd.Series(y_kernel_train).value_counts())

Kernel training shape: (5000, 194)
1    3403
0    1597
Name: count, dtype: int64


In [13]:
from sklearn.metrics import pairwise_distances

def laplacian_kernel(X, Y):
    gamma = 0.05

    distance = pairwise_distances(
        X,
        Y,
        metric="manhattan"
    )

    return np.exp(-gamma * distance)

In [14]:
from sklearn.svm import SVC

custom_svm = SVC(
    kernel=laplacian_kernel,
    class_weight="balanced",
    cache_size=4096
)

In [15]:
custom_svm.fit(
    X_kernel_train,
    y_kernel_train
)

SVC(cache_size=4096, class_weight='balanced',
    kernel=<function laplacian_kernel at 0x7b5e617e2660>)

In [16]:
X_kernel_eval = X_kernel_test[:10000]
y_kernel_eval = y_kernel_test[:10000]

custom_pred = custom_svm.predict(X_kernel_eval)
custom_scores = custom_svm.decision_function(X_kernel_eval)

print("Prediction completed.")

Prediction completed.


In [18]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

In [19]:
custom_accuracy = accuracy_score(
    y_kernel_eval,
    custom_pred
)

custom_precision = precision_score(
    y_kernel_eval,
    custom_pred
)

custom_recall = recall_score(
    y_kernel_eval,
    custom_pred
)

custom_f1 = f1_score(
    y_kernel_eval,
    custom_pred
)

custom_roc_auc = roc_auc_score(
    y_kernel_eval,
    custom_scores
)

custom_pr_auc = average_precision_score(
    y_kernel_eval,
    custom_scores
)

print("===== CUSTOM LAPLACIAN SVM =====")
print(f"Accuracy : {custom_accuracy:.4f}")
print(f"Precision: {custom_precision:.4f}")
print(f"Recall   : {custom_recall:.4f}")
print(f"F1       : {custom_f1:.4f}")
print(f"ROC-AUC  : {custom_roc_auc:.4f}")
print(f"PR-AUC   : {custom_pr_auc:.4f}")

===== CUSTOM LAPLACIAN SVM =====
Accuracy : 0.9404
Precision: 0.9910
Recall   : 0.9475
F1       : 0.9688
ROC-AUC  : 0.9219
PR-AUC   : 0.9976


In [23]:
custom_kernel_path = f"{BASE_PATH}/models/custom_laplacian_svm.pkl"
import joblib
joblib.dump(
    custom_svm,
    custom_kernel_path
)

print("Saved:", custom_kernel_path)

Saved: /content/drive/MyDrive/NetShield-AI/models/custom_laplacian_svm.pkl


In [24]:
import os

print(os.path.exists(custom_kernel_path))

True


In [25]:
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

# Same 10,000 test samples for every binary model
eval_idx, _ = train_test_split(
    np.arange(len(y_test)),
    train_size=10000,
    stratify=y_test,
    random_state=42
)

X_eval = X_test.iloc[eval_idx]
y_eval = y_test.iloc[eval_idx]

# Corresponding transformed data for custom kernel
X_kernel_eval_common = X_kernel_test[eval_idx]
y_kernel_eval_common = y_kernel_test[eval_idx]

print("Common evaluation set:", X_eval.shape)
print("Attack / Benign:")
print(y_eval.value_counts())

Common evaluation set: (10000, 42)
Attack / Benign:
label
1    5506
0    4494
Name: count, dtype: int64


In [27]:
import joblib

logistic_path = f"{BASE_PATH}/models/logistic_binary.pkl"

logistic_model = joblib.load(logistic_path)

print("Logistic model loaded successfully.")

Logistic model loaded successfully.


In [28]:
from sklearn.model_selection import train_test_split
import numpy as np

eval_idx, _ = train_test_split(
    np.arange(len(y_test)),
    train_size=10000,
    stratify=y_test,
    random_state=42
)

X_eval = X_test.iloc[eval_idx]
y_eval = y_test.iloc[eval_idx]

print("Evaluation set:", X_eval.shape)

Evaluation set: (10000, 42)


In [29]:
log_pred = logistic_model.predict(X_eval)
log_score = logistic_model.predict_proba(X_eval)[:, 1]

print("Logistic prediction completed.")

Logistic prediction completed.


In [30]:
print("Logistic:", "logistic_model" in globals())
print("Linear:", "linear_svm" in globals())
print("RBF:", "rbf_svm" in globals())
print("Polynomial:", "poly_svm" in globals())
print("Custom:", "custom_svm" in globals())

Logistic: True
Linear: False
RBF: False
Polynomial: False
Custom: True


In [31]:
import os
import joblib

model_dir = f"{BASE_PATH}/models"

model_files = {
    "linear": f"{model_dir}/linear_svm.pkl",
    "rbf": f"{model_dir}/rbf_svm.pkl",
    "poly": f"{model_dir}/polynomial_svm.pkl"
}

for name, path in model_files.items():
    print(f"{name}: {os.path.exists(path)}")

if os.path.exists(model_files["linear"]):
    linear_svm = joblib.load(model_files["linear"])

if os.path.exists(model_files["rbf"]):
    rbf_svm = joblib.load(model_files["rbf"])

if os.path.exists(model_files["poly"]):
    poly_svm = joblib.load(model_files["poly"])

print("\nLoaded models:")
print("Linear:", "linear_svm" in globals())
print("RBF:", "rbf_svm" in globals())
print("Polynomial:", "poly_svm" in globals())

linear: True
rbf: True
poly: True

Loaded models:
Linear: True
RBF: True
Polynomial: True


In [32]:
# Logistic Regression
log_pred = logistic_model.predict(X_eval)
log_score = logistic_model.predict_proba(X_eval)[:, 1]

# Linear SVM
linear_pred = linear_svm.predict(X_eval)
linear_score = linear_svm.decision_function(X_eval)

# RBF SVM
rbf_pred = rbf_svm.predict(X_eval)
rbf_score = rbf_svm.decision_function(X_eval)

# Polynomial SVM
poly_pred = poly_svm.predict(X_eval)
poly_score = poly_svm.decision_function(X_eval)

# Custom Laplacian SVM
custom_pred_common = custom_svm.predict(X_kernel_eval_common)
custom_score_common = custom_svm.decision_function(
    X_kernel_eval_common
)

print("All predictions completed.")

All predictions completed.


In [33]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

def calculate_binary_metrics(y_true, y_pred, scores):

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, scores),
        "PR-AUC": average_precision_score(y_true, scores)
    }


comparison = []

models_data = [
    ("Logistic Regression", log_pred, log_score, 175341),
    ("Linear SVM", linear_pred, linear_score, 175341),
    ("RBF SVM", rbf_pred, rbf_score, 20000),
    ("Polynomial SVM", poly_pred, poly_score, 20000),
    ("Custom Laplacian SVM", custom_pred_common, custom_score_common, 5000)
]

for name, pred, score, train_size in models_data:

    metrics = calculate_binary_metrics(
        y_eval,
        pred,
        score
    )

    metrics["Model"] = name
    metrics["Training Samples"] = train_size

    comparison.append(metrics)

binary_comparison = pd.DataFrame(comparison)

binary_comparison = binary_comparison[
    [
        "Model",
        "Training Samples",
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC",
        "PR-AUC"
    ]
]

binary_comparison

,Model,Training Samples,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
0,Logistic Regression,175341,0.8415,0.808400,0.933345,0.866391,0.956876,0.968194
1,Linear SVM,175341,0.8361,0.800934,0.934617,0.862627,0.955188,0.967143
2,RBF SVM,20000,0.8505,0.807074,0.957319,0.875800,0.955658,0.962547
3,Polynomial SVM,20000,0.8390,0.785129,0.974210,0.869509,0.940041,0.920227
4,Custom Laplacian SVM,5000,0.8759,0.844118,0.950054,0.893959,0.951613,0.944316


In [34]:
comparison_path = (
    f"{BASE_PATH}/models/binary_model_comparison.csv"
)

binary_comparison.to_csv(
    comparison_path,
    index=False
)

print("Saved:", comparison_path)

Saved: /content/drive/MyDrive/NetShield-AI/models/binary_model_comparison.csv


In [37]:
# Get fitted preprocessing and classifier
log_preprocessor = logistic_model.named_steps["preprocessor"]
log_classifier = logistic_model.named_steps["classifier"]

# Feature names after preprocessing
feature_names = log_preprocessor.get_feature_names_out()

# Choose a highly suspicious test flow
y_prob = logistic_model.predict_proba(X_test)[:, 1]

explain_index = np.argmax(y_prob)

X_explain = X_test.iloc[[explain_index]]

# Transform the selected flow
X_explain_transformed = log_preprocessor.transform(
    X_explain
)

# Convert one row to dense if necessary
if hasattr(X_explain_transformed, "toarray"):
    x_values = X_explain_transformed.toarray().ravel()
else:
    x_values = np.asarray(
        X_explain_transformed
    ).ravel()

# Logistic coefficients
coefficients = log_classifier.coef_[0]

# Feature contributions
contributions = x_values * coefficients

explanation_df = pd.DataFrame({
    "Feature": feature_names,
    "Value": x_values,
    "Coefficient": coefficients,
    "Contribution": contributions
})

# Largest positive contributions
top_positive = explanation_df.sort_values(
    "Contribution",
    ascending=False
).head(10)

print("===== TOP ATTACK CONTRIBUTIONS =====")
print(
    top_positive[
        ["Feature", "Value", "Coefficient", "Contribution"]
    ].to_string(index=False)
)

===== TOP ATTACK CONTRIBUTIONS =====
             Feature      Value  Coefficient  Contribution
    num__trans_depth 168.480595     0.743955    125.341975
           num__dttl   1.560002     9.922568     15.479226
          num__dload  -0.275418   -15.683236      4.319444
           num__sttl   0.723268     1.418166      1.025714
     num__ct_srv_dst  -0.381220    -1.835540      0.699745
          num__sloss   1.030939     0.634478      0.654108
      cat__state_FIN   1.000000     0.410373      0.410373
num__is_sm_ips_ports  -0.126508    -2.285522      0.289137
          num__dpkts   0.063763     4.313425      0.275038
         num__ackdat   2.438706     0.107502      0.262165


In [38]:
import joblib

multiclass_logistic = joblib.load(
    f"{BASE_PATH}/models/multiclass_logistic.pkl"
)

print("Multiclass model loaded.")

Multiclass model loaded.


In [39]:
predicted_attack_type = multiclass_logistic.predict(X_test)

print(predicted_attack_type[:20])

['Shellcode' 'Worms' 'Fuzzers' 'Fuzzers' 'Fuzzers' 'Fuzzers' 'Fuzzers'
 'Fuzzers' 'Normal' 'Normal' 'Normal' 'Normal' 'Fuzzers' 'Fuzzers' 'Worms'
 'Shellcode' 'Shellcode' 'Worms' 'Shellcode' 'Worms']


In [40]:
attack_weights = {
    "Normal": 0.00,
    "Reconnaissance": 0.40,
    "DoS": 0.90,
    "Exploits": 0.85,
    "Backdoor": 0.95,
    "Worms": 1.00,

    # Project-defined defaults for remaining UNSW categories
    "Generic": 0.50,
    "Fuzzers": 0.50,
    "Analysis": 0.40,
    "Shellcode": 0.85
}

attack_type_score = np.array([
    attack_weights.get(label, 0.50)
    for label in predicted_attack_type
])

print(attack_type_score[:20])

[0.85 1.   0.5  0.5  0.5  0.5  0.5  0.5  0.   0.   0.   0.   0.5  0.5
 1.   0.85 0.85 1.   0.85 1.  ]


In [41]:
flow_bytes = (
    test["sbytes"].to_numpy()
    + test["dbytes"].to_numpy()
)

print(flow_bytes[:10])

[ 496 1762 1068  900 2126  784 1960 1384   46   46]


In [42]:
train_flow_bytes = (
    train["sbytes"].to_numpy()
    + train["dbytes"].to_numpy()
)

volume_baseline = np.percentile(
    train_flow_bytes,
    95
)

volume_score = np.clip(
    flow_bytes / volume_baseline,
    0,
    1
)

print("Volume baseline:", volume_baseline)
print("Volume scores:", volume_score[:10])

Volume baseline: 61296.0
Volume scores: [0.00809188 0.02874576 0.01742365 0.01468285 0.03468416 0.01279039
 0.03197599 0.02257896 0.00075046 0.00075046]


In [43]:
repetition_value = np.maximum(
    test["ct_src_ltm"].to_numpy(),
    test["ct_dst_ltm"].to_numpy()
)

train_repetition = np.maximum(
    train["ct_src_ltm"].to_numpy(),
    train["ct_dst_ltm"].to_numpy()
)

repetition_baseline = np.percentile(
    train_repetition,
    95
)

repetition_score = np.clip(
    repetition_value / repetition_baseline,
    0,
    1
)

print("Repetition baseline:", repetition_baseline)
print("Repetition scores:", repetition_score[:10])

Repetition baseline: 26.0
Repetition scores: [0.03846154 0.03846154 0.03846154 0.07692308 0.07692308 0.07692308
 0.07692308 0.03846154 0.07692308 0.07692308]


In [44]:
attack_probability = logistic_model.predict_proba(
    X_test
)[:, 1]

In [45]:
severity_score = 100 * (
    0.60 * attack_probability
    + 0.20 * attack_type_score
    + 0.10 * volume_score
    + 0.10 * repetition_score
)

severity_score = np.clip(
    severity_score,
    0,
    100
)

print(severity_score[:10])

[63.77892042 65.62040295 54.82337159 57.64721067 55.51110305 60.40901087
 59.32484899 52.16364733  0.77673539  0.77673539]


In [47]:
def get_severity_level(score):

    if score < 25:
        return "Informational"

    elif score < 50:
        return "Low"

    elif score < 75:
        return "High"

    else:
        return "Critical"


severity_level = np.array([
    get_severity_level(score)
    for score in severity_score
])

print(severity_level[:20])

['High' 'High' 'High' 'High' 'High' 'High' 'High' 'High' 'Informational'
 'Informational' 'Informational' 'Informational' 'High' 'High' 'High'
 'High' 'High' 'High' 'High' 'High']


In [48]:
alerts = pd.DataFrame({
    "flow_index": np.arange(len(test)),
    "prediction": np.where(
        attack_probability >= 0.5,
        "Attack",
        "Benign"
    ),
    "attack_probability": attack_probability,
    "attack_type": predicted_attack_type,
    "flow_bytes": flow_bytes,
    "repetition_score": repetition_score,
    "severity_score": severity_score,
    "severity": severity_level
})

alerts.head(10)

,flow_index,prediction,attack_probability,attack_type,flow_bytes,repetition_score,severity_score,severity
0,0,Attack,7.718898e-01,Shellcode,496,0.038462,63.778920,High
1,1,Attack,7.491388e-01,Worms,1762,0.038462,65.620403,High
2,2,Attack,7.377420e-01,Fuzzers,1068,0.038462,54.823372,High
3,3,Attack,7.788525e-01,Fuzzers,900,0.076923,57.647211,High
4,4,Attack,7.399172e-01,Fuzzers,2126,0.076923,55.511103,High
5,5,Attack,8.251979e-01,Fuzzers,784,0.076923,60.409011,High
6,6,Attack,8.039310e-01,Fuzzers,1960,0.076923,59.324849,High
7,7,Attack,6.925540e-01,Fuzzers,1384,0.038462,52.163647,High
8,8,Benign,8.733806e-10,Normal,46,0.076923,0.776735,Informational
9,9,Benign,8.733800e-10,Normal,46,0.076923,0.776735,Informational


In [49]:
alerts_display = alerts.copy()

alerts_display["attack_probability"] = (
    alerts_display["attack_probability"] * 100
).round(2)

alerts_display["severity_score"] = (
    alerts_display["severity_score"]
).round(2)

alerts_display["flow_bytes"] = (
    alerts_display["flow_bytes"].astype(int)
)

alerts_display.rename(columns={
    "attack_probability": "attack_probability_%"
}, inplace=True)

alerts_display.head(10)

,flow_index,prediction,attack_probability_%,attack_type,flow_bytes,repetition_score,severity_score,severity
0,0,Attack,77.19,Shellcode,496,0.038462,63.78,High
1,1,Attack,74.91,Worms,1762,0.038462,65.62,High
2,2,Attack,73.77,Fuzzers,1068,0.038462,54.82,High
3,3,Attack,77.89,Fuzzers,900,0.076923,57.65,High
4,4,Attack,73.99,Fuzzers,2126,0.076923,55.51,High
5,5,Attack,82.52,Fuzzers,784,0.076923,60.41,High
6,6,Attack,80.39,Fuzzers,1960,0.076923,59.32,High
7,7,Attack,69.26,Fuzzers,1384,0.038462,52.16,High
8,8,Benign,0.00,Normal,46,0.076923,0.78,Informational
9,9,Benign,0.00,Normal,46,0.076923,0.78,Informational


In [51]:
import os

processed_dir = f"{BASE_PATH}/data/processed"

os.makedirs(processed_dir, exist_ok=True)

alerts_path = f"{processed_dir}/alerts.csv"

alerts.to_csv(
    alerts_path,
    index=False
)

print("Alerts saved to:", alerts_path)

Alerts saved to: /content/drive/MyDrive/NetShield-AI/data/processed/alerts.csv


In [52]:
import os

print(os.path.exists(alerts_path))

True


In [53]:
alerts_path = f"{BASE_PATH}/data/processed/alerts.csv"

alerts.to_csv(
    alerts_path,
    index=False
)

print("Alerts saved to:", alerts_path)

Alerts saved to: /content/drive/MyDrive/NetShield-AI/data/processed/alerts.csv


In [54]:
import time

def simulate_traffic(alerts, delay=1, num_events=20):

    events = alerts.head(num_events)

    for _, event in events.iterrows():

        print("=" * 60)

        print(f"Flow ID          : {event['flow_index']}")
        print(f"Prediction       : {event['prediction']}")
        print(
            f"Attack Probability: "
            f"{event['attack_probability'] * 100:.2f}%"
        )
        print(f"Attack Type      : {event['attack_type']}")
        print(f"Flow Bytes       : {int(event['flow_bytes'])}")
        print(f"Severity Score   : {event['severity_score']:.2f}")
        print(f"Severity         : {event['severity']}")

        time.sleep(delay)

In [55]:
simulate_traffic(
    alerts,
    delay=1,
    num_events=20
)

Flow ID          : 0
Prediction       : Attack
Attack Probability: 77.19%
Attack Type      : Shellcode
Flow Bytes       : 496
Severity Score   : 63.78
Severity         : High
Flow ID          : 1
Prediction       : Attack
Attack Probability: 74.91%
Attack Type      : Worms
Flow Bytes       : 1762
Severity Score   : 65.62
Severity         : High
Flow ID          : 2
Prediction       : Attack
Attack Probability: 73.77%
Attack Type      : Fuzzers
Flow Bytes       : 1068
Severity Score   : 54.82
Severity         : High
Flow ID          : 3
Prediction       : Attack
Attack Probability: 77.89%
Attack Type      : Fuzzers
Flow Bytes       : 900
Severity Score   : 57.65
Severity         : High
Flow ID          : 4
Prediction       : Attack
Attack Probability: 73.99%
Attack Type      : Fuzzers
Flow Bytes       : 2126
Severity Score   : 55.51
Severity         : High
Flow ID          : 5
Prediction       : Attack
Attack Probability: 82.52%
Attack Type      : Fuzzers
Flow Bytes       : 784
Severity

In [56]:
import sklearn
import joblib

print("scikit-learn:", sklearn.__version__)
print("joblib:", joblib.__version__)

scikit-learn: 1.6.1
joblib: 1.6.0
